In [27]:
# Installera RWWA kodpaket till R
devtools::install_github("WorldWeatherAttribution/rwwa")
library(rwwa)
library(extRemes)

Skipping install of 'rwwa' from a github remote, the SHA1 (ad29462b) has not changed since last install.
  Use `force = TRUE` to force installation



* Ladda in datamängderna

In [29]:
# Ladda in alla dataset och lägga in gmst i dem, 
# göra "under-sets" utan inkludering av 2024
gmst <- read.csv("/Users/joel/Livet/Globala System/År 3/LP 3/Kandidatarbete/Newell-Matthew-CCDetectonAttribution_Esbjerg-ac778b5/files/Rx1day-files/annual_sampling/observations/GMST/GMST_anomalies.csv", col.names = c("year", "GMST"), sep = ";", header = TRUE)
setwd("/Users/joel/Livet/Globala System/År 3/LP 3/Kandidatarbete/Kandidatarbete kod/Analyser mapp/Färdiga resultat")

df_C <- read.csv("Färdig_data_fler_punkter.csv") # 1961 data
df_B <- read.csv("data_1920_2025.csv") # 1920 data
df_A <- read.csv("Block_maxima_1874_2025.csv") # 1874 data
df_Nordby <- read.csv("Nordby_1874_2025_max.csv") # Nordby 1874 data

df_C1 <- merge(df_C,gmst, by = "year")
df_C2 <- df_C1[df_C1$year <= 2023, ] 

df_B1 <- merge(df_B,gmst, by = "year")
df_B2 <- df_B1[df_B1$year <= 2023, ] 

df_A1 <- merge(df_A,gmst, by = "year")
df_A2 <- df_A1[df_A1$year <= 2023, ] 

df_Nordby<- df_Nordby[, -1]
df_Nordby1 <- merge(df_Nordby, gmst, by = "year")
df_Nordby2 <- df_Nordby1[df_Nordby1$year <= 2023, ] 

* Gör modeller för vardera dataset, stationär samt med trenden WWA använder sig av

In [34]:
# Använder oss av type = "fixed-disp" då detta gör att modellen använder sig av parameterekvationerna för WWA vi nämner i rapporten

mdl_stat_C1 <- fit_ns(dist="gev", type="fixeddisp", data=df_C1, varnm="rain", covnm=NA, lower = F) #c("GMST")
mdl_stat_C2 <- fit_ns(dist="gev", type="fixeddisp", data=df_C2, varnm="rain", covnm=NA, lower = F) #c("GMST")
mdl_C1 <- fit_ns(dist="gev", type="fixeddisp", data=df_C1, varnm="rain", covnm=c("GMST"), lower = F) #c("GMST")
mdl_C2 <- fit_ns(dist="gev", type="fixeddisp", data=df_C2, varnm="rain", covnm=c("GMST"), lower = F) #c("GMST")


mdl_stat_B1 <- fit_ns(dist="gev", type="fixeddisp", data=df_B1, varnm="rain", covnm=NA, lower = F) #c("GMST")
mdl_stat_B2 <- fit_ns(dist="gev", type="fixeddisp", data=df_B2, varnm="rain", covnm=NA, lower = F) #c("GMST")
mdl_B1 <- fit_ns(dist="gev", type="fixeddisp", data=df_B1, varnm="rain", covnm=c("GMST"), lower = F) #c("GMST")
mdl_B2 <- fit_ns(dist="gev", type="fixeddisp", data=df_B2, varnm="rain", covnm=c("GMST"), lower = F) #c("GMST")

mdl_stat_A1 <- fit_ns(dist="gev", type="fixeddisp", data=df_A1, varnm="rain", covnm=NA, lower = F) #c("GMST")
mdl_stat_A2 <- fit_ns(dist="gev", type="fixeddisp", data=df_A2, varnm="rain", covnm=NA, lower = F) #c("GMST")
mdl_A1 <- fit_ns(dist="gev", type="fixeddisp", data=df_A1, varnm="rain", covnm=c("GMST"), lower = F) #c("GMST")
mdl_A2 <- fit_ns(dist="gev", type="fixeddisp", data=df_A2, varnm="rain", covnm=c("GMST"), lower = F) #c("GMST")

mdl_stat_Nordby1 <- fit_ns(dist="gev", type="fixeddisp", data=df_Nordby1, varnm="rain", covnm=NA, lower = F) #c("GMST")
mdl_stat_Nordby2 <- fit_ns(dist="gev", type="fixeddisp", data=df_Nordby2, varnm="rain", covnm=NA, lower = F) #c("GMST")
mdl_Nordby1 <- fit_ns(dist="gev", type="fixeddisp", data=df_Nordby1, varnm="rain", covnm=c("GMST"), lower = F) #c("GMST")
mdl_Nordby2 <- fit_ns(dist="gev", type="fixeddisp", data=df_Nordby2, varnm="rain", covnm=c("GMST"), lower = F) #c("GMST")

* Skapar funktion för att beräkna p-värde med WWA-modell

In [31]:
p_value <- function (mdl_stat,mdl_nonstat){
    
    logL_null <- -mdl_stat$value
    logL_alt  <- -mdl_nonstat$value # Din modell med covnm = "GMST"
    
    # Beräkna D
    D <- 2 * (logL_alt - logL_null)
    
    # Bestäm frihetsgrader (df)
    # Skillnaden är 1 parameter (alpha_GMST)
    df_diff <- 1 
    
    # Beräkna p-värdet
    p_val <- 1 - pchisq(D, df = df_diff)

    return (p_val)
    }


In [35]:
cat("A1:")
p_value(mdl_stat_A1,mdl_A1)
cat("A2:")
p_value(mdl_stat_A2,mdl_A2)
cat("B1:")
p_value(mdl_stat_B1,mdl_B1)
cat("B2:")
p_value(mdl_stat_B2,mdl_B2)
cat("C1:")
p_value(mdl_stat_C1,mdl_C1)
cat("C2:")
p_value(mdl_stat_C2,mdl_C2)
cat("Nordby1:")
p_value(mdl_stat_Nordby1,mdl_Nordby1)
cat("Nordby2:")
p_value(mdl_stat_Nordby2,mdl_Nordby2)

A1:

[1] 0.1472959

A2:

[1] 0.2314177

B1:

...

[1] 0.1228985

Nordby1:

[1] 0.09666435

Nordby2:

[1] 0.19151

* Skapar funktion som beräknar P0 och P1

In [8]:
p <- function(mdl){
    P1 <- 1/mdl[7] # mdl[7] motsvarar return period, 1/RP = P1
    P0 <- P1 / mdl[8] # mdl[8] motsvarar PR, P1/PR = P0
    return(list("P0" = round(P0, 8), "P1" = round(P1, 8)))
    }

* Plockar ut GMST för enskilda år

In [22]:
# Plocka ut enskilda GMST för 1960 samt 2024 för kovariat
cov_1874 <- gmst[gmst$year == 1874,c("GMST"),drop = F]
cov_1900 <- gmst[gmst$year == 1900,c("GMST"),drop = F]
cov_1920 <- gmst[gmst$year == 1920,c("GMST"),drop = F]
cov_1961 <- gmst[gmst$year == 1961,c("GMST"),drop = F]
cov_2024 <- gmst[gmst$year == 2024,c("GMST"),drop = F]

* Kör RWWA:s funktion boot_ci med tröskelvärde 144.6

In [42]:
res_A1_1874_144.6 <- boot_ci(mdl_A1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 144.6,
                   nsamp  = 10)
res_A1_1961_144.6 <- boot_ci(mdl_A1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 144.6,
                   nsamp  = 10)

res_A2_1874_144.6  <- boot_ci(mdl_A2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 144.6,
                   nsamp  = 10)
res_A2_1961_144.6 <- boot_ci(mdl_A2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 144.6,
                   nsamp  = 10)

res_B1_1920_144.6  <- boot_ci(mdl_B1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1920,
                   ev     = 144.6,
                   nsamp  = 10)
res_B1_1961_144.6 <- boot_ci(mdl_B1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 144.6,
                   nsamp  = 10)

res_B2_1920_144.6  <- boot_ci(mdl_B2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1920,
                   ev     = 144.6,
                   nsamp  = 10)

res_B2_1961_144.6 <- boot_ci(mdl_B2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 144.6,
                   nsamp  = 10)

res_C1_1961_144.6  <- boot_ci(mdl_C1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 144.6,
                   nsamp  = 10)

res_C2_1961_144.6 <- boot_ci(mdl_C2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 144.6,
                   nsamp  = 10)

res_Nordby1_1874_144.6 <- boot_ci(mdl_Nordby1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 144.6,
                   nsamp  = 10)

res_Nordby1_1961_144.6 <- boot_ci(mdl_Nordby1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 144.6,
                   nsamp  = 10)


res_Nordby2_1874_144.6 <- boot_ci(mdl_Nordby2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 144.6,
                   nsamp  = 10)

res_Nordby2_1961_144.6 <- boot_ci(mdl_Nordby2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 144.6,
                   nsamp  = 10)

* Kör RWWA:s funktion boot_ci med tröskelvärde 97.6

In [38]:
res_A1_1874_97.6 <- boot_ci(mdl_A1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 97.6,
                   nsamp  = 10)
res_A1_1961_97.6 <- boot_ci(mdl_A1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 97.6,
                   nsamp  = 10)

res_A2_1874_97.6  <- boot_ci(mdl_A2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 97.6,
                   nsamp  = 10)
res_A2_1961_97.6 <- boot_ci(mdl_A2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 97.6,
                   nsamp  = 10)

res_B1_1920_97.6  <- boot_ci(mdl_B1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1920,
                   ev     = 97.6,
                   nsamp  = 10)
res_B1_1961_97.6 <- boot_ci(mdl_B1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 97.6,
                   nsamp  = 10)

res_B2_1920_97.6  <- boot_ci(mdl_B2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1920,
                   ev     = 97.6,
                   nsamp  = 10)

res_B2_1961_97.6 <- boot_ci(mdl_B2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 97.6,
                   nsamp  = 10)

res_C1_1961_97.6  <- boot_ci(mdl_C1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 97.6,
                   nsamp  = 10)

res_C2_1961_97.6 <- boot_ci(mdl_C2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 97.6,
                   nsamp  = 10)

res_Nordby1_1874_97.6 <- boot_ci(mdl_Nordby1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 97.6,
                   nsamp  = 10)

res_Nordby1_1961_97.6 <- boot_ci(mdl_Nordby1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 97.6,
                   nsamp  = 10)


res_Nordby2_1874_97.6 <- boot_ci(mdl_Nordby2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 97.6,
                   nsamp  = 10)

res_Nordby2_1961_97.6 <- boot_ci(mdl_Nordby2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 97.6,
                   nsamp  = 10)

* Kör RWWA:s funktion boot_ci med tröskelvärde 30

In [39]:
res_A1_1874_30 <- boot_ci(mdl_A1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 30,
                   nsamp  = 10)

res_A1_1961_30 <- boot_ci(mdl_A1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 30,
                   nsamp  = 10)

res_A2_1874_30 <- boot_ci(mdl_A2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 30,
                   nsamp  = 10)

res_A2_1961_30 <- boot_ci(mdl_A2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 30,
                   nsamp  = 10)

res_B1_1920_30 <- boot_ci(mdl_B1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1920,
                   ev     = 30,
                   nsamp  = 10)

res_B1_1961_30 <- boot_ci(mdl_B1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 30,
                   nsamp  = 10)

res_B2_1920_30 <- boot_ci(mdl_B2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1920,
                   ev     = 30,
                   nsamp  = 10)

res_B2_1961_30 <- boot_ci(mdl_B2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 30,
                   nsamp  = 10)

res_C1_1961_30 <- boot_ci(mdl_C1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 30,
                   nsamp  = 10)

res_C2_1961_30 <- boot_ci(mdl_C2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 30,
                   nsamp  = 10)

res_Nordby1_1874_30 <- boot_ci(mdl_Nordby1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 30,
                   nsamp  = 10)

res_Nordby1_1961_30 <- boot_ci(mdl_Nordby1,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 30,
                   nsamp  = 10)


res_Nordby2_1874_30 <- boot_ci(mdl_Nordby2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1874,
                   ev     = 30,
                   nsamp  = 10)

res_Nordby2_1961_30 <- boot_ci(mdl_Nordby2,
                   cov_f  = cov_2024,
                   cov_cf = cov_1961,
                   ev     = 30,
                   nsamp  = 10)

* Threshold 144.6, 2024 ej inkluderat

In [43]:
cat("-----------------------------------------------------\nDataset A:\n")
cat("res_A2_1874_144.6")
res_A2_1874_144.6
p(res_A2_1874_144.6)

cat("res_A2_1961_144.6")
res_A2_1961_144.6
p(res_A2_1961_144.6)

cat("-----------------------------------------------------\nDataset B:\n")
cat("res_B2_1920_144.6")
res_B2_1920_144.6
p(res_B2_1920_144.6)

cat("res_B2_1961_144.6")
res_B2_1961_144.6
p(res_B2_1961_144.6)

cat("-----------------------------------------------------\nDataset C:\n")
cat("res_C2_1961_144.6")
res_C2_1961_144.6
p(res_C2_1961_144.6)

cat("-----------------------------------------------------\nDataset Nordby:\n")
cat("res_Nordby2_1874_144.6")
res_Nordby2_1874_144.6
p(res_Nordby2_1874_144.6)

cat("res_Nordby2_1961_144.6")
res_Nordby2_1961_144.6
p(res_Nordby2_1961_144.6)

-----------------------------------------------------
Dataset A:
res_A2_1874_144.6

,est,2.5%,97.5%
mu0,37.607936,35.68709252,39.3876708
sigma0,9.042339,8.47463974,9.9940158
alpha_GMST,2.509804,-1.68895669,6.0318562
shape,0.168554,0.06758903,0.2597706
disp,0.240437,0.23579905,0.2578383
event_magnitude,144.600000,144.60000000,144.6000000
return_period,418.509580,137.55371916,3566.5766182
PR,1.593332,0.72156569,3.5796795
dI_abs,12.142708,-8.69218032,28.3715295
dI_rel,9.167263,-5.60832081,24.4498612


$P0
[1] 0.00149964

$P1
[1] 0.00238943

res_A2_1961_144.6

,est,2.5%,97.5%
mu0,37.607936,35.68709252,39.3876708
sigma0,9.042339,8.47463974,9.9940158
alpha_GMST,2.509804,-1.68895669,6.0318562
shape,0.168554,0.06758903,0.2597706
disp,0.240437,0.23579905,0.2578383
event_magnitude,144.600000,144.60000000,144.6000000
return_period,418.509580,137.55371916,3566.5766182
PR,1.437573,0.77379119,2.6860489
dI_abs,9.561675,-6.72757324,22.6530851
dI_rel,7.080712,-4.40748973,18.5992174


...

,est,2.5%,97.5%
mu0,26.8091116,25.5339804,2.817081e+01
sigma0,6.8804938,5.8751969,8.048719e+00
alpha_GMST,2.1008049,0.1723770,3.506163e+00
shape,0.1141383,0.1166450,2.066574e-01
disp,0.2566476,0.2227598,2.879062e-01
event_magnitude,144.6000000,144.6000000,1.446000e+02
return_period,6354.0631103,908.8206948,1.089309e+04
PR,2.0653001,1.0501857,2.385828e+00
dI_abs,14.1512005,1.1874826,2.378534e+01
dI_rel,10.8480879,0.8342073,1.969136e+01


$P0
[1] 7.62e-05

$P1
[1] 0.00015738

res_Nordby2_1961_144.6

,est,2.5%,97.5%
mu0,26.8091116,25.5339804,2.817081e+01
sigma0,6.8804938,5.8751969,8.048719e+00
alpha_GMST,2.1008049,0.1723770,3.506163e+00
shape,0.1141383,0.1166450,2.066574e-01
disp,0.2566476,0.2227598,2.879062e-01
event_magnitude,144.6000000,144.6000000,1.446000e+02
return_period,6354.0631103,908.8206948,1.089309e+04
PR,1.7584747,1.0386926,1.968793e+00
dI_abs,11.1614615,0.9278018,1.891286e+01
dI_rel,8.3644962,0.6495349,1.504985e+01


$P0
[1] 8.95e-05

$P1
[1] 0.00015738

* Threshold 144.6, 2024 inkluderat

In [382]:
cat("-----------------------------------------------------\nDataset A:\n")
cat("1874")
res_A1_1874_144.6
p(res_A1_1874_144.6)
cat("1961")
res_A1_1961_144.6
p(res_A1_1961_144.6)
cat("-----------------------------------------------------\nDataset B:\n")
cat("1920")
res_B1_1920_144.6
p(res_B1_1920_144.6)
cat("1961")
res_B1_1961_144.6
p(res_B1_1961_144.6)
cat("-----------------------------------------------------\nDataset C:\n")
cat("1874")
res_C1_1961_144.6
p(res_C1_1961_144.6)
cat("-----------------------------------------------------\nDataset Nordby:\n")
cat("1874")
res_Nordby1_1874_144.6
p(res_Nordby1_1874_144.6)
cat("1961")
res_Nordby1_1961_144.6
p(res_Nordby1_1961_144.6)

-----------------------------------------------------
Dataset A:
1874

,est,2.5%,97.5%
mu0,37.5123280,35.63600101,39.7018732
sigma0,9.1219163,7.71003794,10.3037228
alpha_GMST,3.0290736,-1.12623301,7.3197636
shape,0.1784464,0.06152933,0.3043190
disp,0.2431712,0.21039020,0.2697998
event_magnitude,144.6000000,144.60000000,144.6000000
return_period,325.6515273,91.09834047,4827.0401383
PR,1.7165768,0.82074144,4.8355720
dI_abs,14.5598326,-5.58271170,33.3988828
dI_rel,11.1964118,-3.71727976,30.0346780


$P0
[1] 0.00179

$P1
[1] 0.00307

1961

,est,2.5%,97.5%
mu0,37.5123280,35.63600101,39.7018732
sigma0,9.1219163,7.71003794,10.3037228
alpha_GMST,3.0290736,-1.12623301,7.3197636
shape,0.1784464,0.06152933,0.3043190
disp,0.2431712,0.21039020,0.2697998
event_magnitude,144.6000000,144.60000000,144.6000000
return_period,325.6515273,91.09834047,4827.0401383
PR,1.5234504,0.85709424,3.3925474
dI_abs,11.4876029,-4.33618211,26.7838928
dI_rel,8.6300022,-2.91143626,22.7336502


...

,est,2.5%,97.5%
mu0,26.6379250,25.07736659,28.3527089
sigma0,6.9421663,5.91712084,7.9175748
alpha_GMST,2.6444281,-0.08512064,5.4401498
shape,0.1431496,0.01439807,0.2636723
disp,0.2606121,0.22619536,0.2911877
event_magnitude,144.6000000,144.60000000,144.6000000
return_period,2495.8280774,374.49101509,1036400.6757652
PR,2.1913248,0.98204983,13.5598007
dI_abs,17.6876198,-0.58230763,35.2519140
dI_rel,13.9368750,-0.40108718,32.2382552


$P0
[1] 0.00018

$P1
[1] 0.0004

1961

,est,2.5%,97.5%
mu0,26.6379250,25.07736659,28.3527089
sigma0,6.9421663,5.91712084,7.9175748
alpha_GMST,2.6444281,-0.08512064,5.4401498
shape,0.1431496,0.01439807,0.2636723
disp,0.2606121,0.22619536,0.2911877
event_magnitude,144.6000000,144.60000000,144.6000000
return_period,2495.8280774,374.49101509,1036400.6757652
PR,1.8416826,0.98597112,7.4382299
dI_abs,13.9915175,-0.45398625,28.3180194
dI_rel,10.7125642,-0.31297743,24.3528876


$P0
[1] 0.00022

$P1
[1] 0.0004

* Threshold 97.6, 2024 ej inkluderat

In [383]:
cat("-----------------------------------------------------\nDataset A:\n")
cat("1874")
res_A2_1874_97.6
p(res_A2_1874_97.6)
cat("1961")
res_A2_1961_97.6
p(res_A2_1961_97.6)
cat("-----------------------------------------------------\nDataset B:\n")
cat("1920")
res_B2_1920_97.6
p(res_B2_1920_97.6)
cat("1961")
res_B2_1961_97.6
p(res_B2_1961_97.6)
cat("-----------------------------------------------------\nDataset C:\n")
cat("1961")
res_C2_1961_97.6
p(res_C2_1961_97.6)
cat("-----------------------------------------------------\nDataset Nordby:\n")
cat("1874")
res_Nordby2_1874_97.6
p(res_Nordby2_1874_97.6)
cat("1961")
res_Nordby2_1961_97.6
p(res_Nordby2_1961_97.6)

-----------------------------------------------------
Dataset A:
1874

,est,2.5%,97.5%
mu0,37.607936,35.73984602,39.7997452
sigma0,9.042339,7.71471492,10.2368435
alpha_GMST,2.509804,-1.74899016,6.8936424
shape,0.168554,0.03318026,0.2889453
disp,0.240437,0.20888525,0.2667397
event_magnitude,97.600000,97.60000000,97.6000000
return_period,55.341124,22.79886420,320.6797959
PR,1.553923,0.71540938,3.7033053
dI_abs,8.195908,-5.98580752,21.7533199
dI_rel,9.167263,-5.77859734,28.6806491


$P0
[1] 0.01163

$P1
[1] 0.01807

1961

,est,2.5%,97.5%
mu0,37.607936,35.73984602,39.7997452
sigma0,9.042339,7.71471492,10.2368435
alpha_GMST,2.509804,-1.74899016,6.8936424
shape,0.168554,0.03318026,0.2889453
disp,0.240437,0.20888525,0.2667397
event_magnitude,97.600000,97.60000000,97.6000000
return_period,55.341124,22.79886420,320.6797959
PR,1.409448,0.77002047,2.7490640
dI_abs,6.453800,-4.63805769,17.4263050
dI_rel,7.080712,-4.53652711,21.7356892


...

,est,2.5%,97.5%
mu0,26.8091116,25.28680119,28.4370179
sigma0,6.8804938,5.88483850,7.8297875
alpha_GMST,2.1008049,-0.67658239,4.7643593
shape,0.1141383,-0.01038921,0.2341188
disp,0.2566476,0.22303355,0.2874453
event_magnitude,97.6000000,97.60000000,97.6000000
return_period,462.3345944,115.07094707,12742.2162253
PR,1.9390468,0.81088896,9.4738199
dI_abs,9.5515710,-3.11717204,20.8703670
dI_rel,10.8480879,-3.09497570,27.1998795


$P0
[1] 0.00112

$P1
[1] 0.00216

1961

,est,2.5%,97.5%
mu0,26.8091116,25.28680119,28.4370179
sigma0,6.8804938,5.88483850,7.8297875
alpha_GMST,2.1008049,-0.67658239,4.7643593
shape,0.1141383,-0.01038921,0.2341188
disp,0.2566476,0.22303355,0.2874453
event_magnitude,97.6000000,97.60000000,97.6000000
return_period,462.3345944,115.07094707,12742.2162253
PR,1.6735112,0.84909942,5.6133179
dI_abs,7.5336005,-2.42289200,16.6992600
dI_rel,8.3644962,-2.42233745,20.6416659


$P0
[1] 0.00129

$P1
[1] 0.00216

* Threshold 97.6, 2024 inkluderat

In [384]:
cat("-----------------------------------------------------\nDataset A:\n")
cat("1874")
res_A1_1874_97.6
p(res_A1_1874_97.6)
cat("1961")
res_A1_1961_97.6
p(res_A1_1961_97.6)
cat("-----------------------------------------------------\nDataset B:\n")
cat("1920")
res_B1_1920_97.6
p(res_B1_1920_97.6)
cat("1961")
res_B1_1961_97.6
p(res_B1_1961_97.6)
cat("-----------------------------------------------------\nDataset C:\n")
cat("1961")
res_C1_1961_97.6
p(res_C1_1961_97.6)
cat("-----------------------------------------------------\nDataset Nordby:\n")
cat("1874")
res_Nordby1_1874_97.6
p(res_Nordby1_1874_97.6)
cat("1961")
res_Nordby1_1961_97.6
p(res_Nordby1_1961_97.6)

-----------------------------------------------------
Dataset A:
1874

,est,2.5%,97.5%
mu0,37.5123280,35.63600101,39.7018732
sigma0,9.1219163,7.71003794,10.3037228
alpha_GMST,3.0290736,-1.12623301,7.3197636
shape,0.1784464,0.06152933,0.3043190
disp,0.2431712,0.21039020,0.2697998
event_magnitude,97.6000000,97.60000000,97.6000000
return_period,46.6393471,20.70508141,183.0814056
PR,1.6720321,0.82898901,4.0383912
dI_abs,9.8273836,-3.76813736,22.5430910
dI_rel,11.1964118,-3.71727976,30.0346780


$P0
[1] 0.01282

$P1
[1] 0.02144

1961

,est,2.5%,97.5%
mu0,37.5123280,35.63600101,39.7018732
sigma0,9.1219163,7.71003794,10.3037228
alpha_GMST,3.0290736,-1.12623301,7.3197636
shape,0.1784464,0.06152933,0.3043190
disp,0.2431712,0.21039020,0.2697998
event_magnitude,97.6000000,97.60000000,97.6000000
return_period,46.6393471,20.70508141,183.0814056
PR,1.4920617,0.86377018,2.9505613
dI_abs,7.7537347,-2.92677299,18.0782015
dI_rel,8.6300022,-2.91143626,22.7336502


...

,est,2.5%,97.5%
mu0,26.6379250,25.07736659,28.3527089
sigma0,6.9421663,5.91712084,7.9175748
alpha_GMST,2.6444281,-0.08512064,5.4401498
shape,0.1431496,0.01439807,0.2636723
disp,0.2606121,0.22619536,0.2911877
event_magnitude,97.6000000,97.60000000,97.6000000
return_period,258.8309488,79.54480617,4549.7611183
PR,2.0841074,0.98246916,8.2671428
dI_abs,11.9385317,-0.39293191,23.7930864
dI_rel,13.9368750,-0.40097985,32.2369351


$P0
[1] 0.00185

$P1
[1] 0.00386

1961

,est,2.5%,97.5%
mu0,26.6379250,25.07736659,28.3527089
sigma0,6.9421663,5.91712084,7.9175748
alpha_GMST,2.6444281,-0.08512064,5.4401498
shape,0.1431496,0.01439807,0.2636723
disp,0.2606121,0.22619536,0.2911877
event_magnitude,97.6000000,97.60000000,97.6000000
return_period,258.8309488,79.54480617,4549.7611183
PR,1.7702736,0.98629936,5.0752324
dI_abs,9.4437905,-0.30634272,19.1130729
dI_rel,10.7125642,-0.31289364,24.3519193


$P0
[1] 0.00218

$P1
[1] 0.00386